# Global estimate

Electricity that rice husk and corncob could supply to the rural population without access (Section 5.2). Production of paddy rice and maize: FAOSTAT, mean of 2022-2024; rural access and rural population: World Bank, World Development Indicators. Both ship with the package in `data/global`.

In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path
ARTICLE = Path.cwd().parent / "results"
OUTPUT = Path.cwd() / "output" / "08_global_potential"
OUTPUT.mkdir(parents=True, exist_ok=True)
if (ARTICLE / "best_vectors.npz").exists():
    handle, temporary = tempfile.mkstemp(dir=OUTPUT, suffix=".npz")
    os.close(handle)
    shutil.copyfile(ARTICLE / "best_vectors.npz", temporary)
    os.replace(temporary, OUTPUT / "best_vectors.npz")
os.environ.setdefault("CCPYRO_RESULTS", str(OUTPUT))
os.environ.setdefault("CCPYRO_FIGDATA", str(OUTPUT / "figdata"))
try:
    import ccpyro  # installed with `pip install -e .`
except ImportError:  # running from a clone without installation
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import ccpyro
import numpy as np
import pandas as pd
np.set_printoptions(precision=6, suppress=True)
print("ccpyro", ccpyro.__version__)

In [ ]:
from ccpyro import global_potential as gp
import matplotlib.pyplot as plt
yields = pd.read_csv(ARTICLE / "product_yields.csv")
feeds = pd.read_csv(ARTICLE / "feedstock_characterisation.csv").set_index("treatment")
summary, countries, e_rh, e_cc = gp.estimate(yields, feeds, 500.0)
print(f"rice husk {np.median(e_rh):.0f} kWh/t, corncob {np.median(e_cc):.0f} kWh/t")
summary[summary.region == "World"].round(3).T

In [ ]:
summary[summary.region != "World"].round(3)

In [ ]:
top = countries[countries.tier == "Tier 3"].sort_values("unserved_M", ascending=False).head(15)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top.iso3[::-1], top.unserved_M[::-1], color="0.8", label="rural population without access")
ax.barh(top.iso3[::-1], top.people_served_M[::-1], color="C0", label="served at Tier 3")
ax.set_xlabel("million people"); ax.legend(fontsize=8)
plt.tight_layout()